# Zero-ablate a layer

Interventions let you fork a captured log, edit one site, and replay downstream computation without rewriting the model.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import torch
import torchlens as tl

shared_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "notebooks" / "total_audit" / "_shared.py"
    if candidate.exists():
        shared_path = candidate
        break
if shared_path is None:
    raise FileNotFoundError("Could not find notebooks/total_audit/_shared.py")

spec = importlib.util.spec_from_file_location("torchlens_total_audit_shared", shared_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {shared_path}")
shared = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = shared
spec.loader.exec_module(shared)
tiny_model = shared.tiny_model

torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(2, 4)

The core intervention is three lines: capture an intervention-ready log, attach a zero-ablation hook, then replay.

In [ ]:
clean = tl.log_forward_pass(model, x, layers_to_save="all", intervention_ready=True)
ablated = clean.fork("zero_relu").attach_hooks(tl.label("relu_1_2"), tl.zero_ablate())
ablated.replay()

The final output changed because the ReLU activation was replaced with zeros before downstream layers ran.

In [ ]:
print("clean output:", clean.layer_list[-1].activation)
print("ablated output:", ablated.layer_list[-1].activation)
print(
    "changed:",
    not torch.allclose(clean.layer_list[-1].activation, ablated.layer_list[-1].activation),
)